<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_soec_multiphysics.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.2 — Steam, Charge and Heat in a Solid Oxide Electrolysis Cell

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L10.2 · Solid Oxide Cells**

A solid oxide electrolysis cell splits steam into hydrogen at 750 to 850 °C.
Along its fuel channel three quantities change together: the **steam
fraction** falls as steam is used, the **local current density** follows the
steam fraction and the temperature, and the **temperature** rises or falls
with the heat the current releases or absorbs. Each depends on the others, so
they must be solved together: this is a **multiphysics** problem.

The notebook has two parts. **Part A** trains one physics-informed network
for every operating point - every cell voltage from 1.15 to 1.35 V and every
inlet temperature from 750 to 850 °C - and measures what it gains over finite
volumes when many operating points are asked for. **Part B** defines the cost
of an hour of operation (electricity, hydrogen sold, ageing of the stack) and
finds the cell voltage that minimises it under a temperature limit, by a
gradient taken through the network.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data, its physics, and why it is multiphysics | — |
| **4** | the reference, and coupled finite volumes for an agreed accuracy | — |
| **5** | Part A: one network for every operating point, and its speed | the two balances, the charge transfer |
| **6** | Part B: the optimal cell voltage, by a gradient through the network | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About three minutes on a CPU with the trained weights; set
`FULL_TRAINING = True` in section 5 to train them again (about 35
minutes on a CPU). Physical values are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py', 'Ex10.2_weights.pt']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex10.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

### The solid oxide electrolyser

A **solid oxide electrolysis cell (SOEC)** splits steam into hydrogen and
oxygen at 700 to 850 °C. The name comes from its electrolyte: a *solid*
ceramic *oxide*, usually zirconia doped with about 8 mol % yttria
(yttria-stabilised zirconia, YSZ). Each pair of Y³⁺ ions that replaces Zr⁴⁺
leaves one oxygen site empty, and oxide ions O²⁻ move through the crystal by
hopping between these vacancies. The hopping is thermally activated, so the
ceramic conducts well enough only above about 700 °C, which is why the cell
runs hot. The electrolyte conducts neither electrons nor gas.

The cell is a stack of thin layers, from the fuel side to the air side:

- **Fuel channel.** Steam with about 10 % hydrogen flows in a channel cut
  into the metal interconnect. The hydrogen keeps the nickel of the electrode
  metallic, and it keeps the Nernst voltage, which contains
  $\ln(p_{\mathrm{H_2}}/p_{\mathrm{H_2O}})$, finite at the inlet.
- **Fuel electrode.** A porous cermet of nickel and YSZ, a few hundred µm
  thick when it also carries the cell mechanically. Nickel conducts electrons
  and catalyses the reaction, YSZ conducts O²⁻, and the pores bring steam in
  and take hydrogen out. Where the three meet, at the triple-phase boundary,
  steam takes two electrons from the power supply:
  $\mathrm{H_2O} + 2e^- \rightarrow \mathrm{H_2} + \mathrm{O^{2-}}$.
- **Electrolyte.** Dense YSZ, 5 to 10 µm thick and gas-tight. It carries the
  O²⁻ to the other side and keeps hydrogen and air apart.
- **Air electrode.** A porous, electron-conducting ceramic oxide (LSM–YSZ, or
  LSCF on a thin ceria barrier layer). It turns the oxide ions back into gas
  and returns the electrons to the external circuit:
  $2\,\mathrm{O^{2-}} \rightarrow \mathrm{O_2} + 4e^-$. Without it the circuit
  would not be closed.
- **Air channel.** Ribs and channels in the interconnect: the ribs collect
  the current, the channels carry air that sweeps the oxygen away, lowers its
  partial pressure and helps manage the heat.

Overall, $\mathrm{H_2O} \rightarrow \mathrm{H_2} + \tfrac12\,\mathrm{O_2}$:
hydrogen leaves on the fuel side, oxygen on the air side, and the two never
mix. The same cell run backwards is a solid oxide fuel cell (SOFC), the cell
of Ex_09.2.

**Compared with a PEM electrolyser.** A PEM cell has the same layout of
channels, two electrodes and a thin electrolyte, but the electrolyte is a
humid polymer membrane that carries protons H⁺ at 50 to 80 °C, so the ions
move the other way. Liquid water is fed to the oxygen side and split there,
$2\,\mathrm{H_2O} \rightarrow \mathrm{O_2} + 4\,\mathrm{H^+} + 4e^-$, and the
protons cross the membrane and form hydrogen at the other electrode. The
hydrogen therefore leaves nearly pure, often at elevated pressure, whereas an
SOEC delivers it mixed with the unused steam, which must be condensed out.
The cold, acidic PEM cell needs platinum and iridium catalysts and follows a
varying load within seconds; the SOEC uses nickel and ceramic oxides and
needs less electricity where heat is available, but must be kept hot. The
table below compares the numbers.

**Why the high temperature pays.** Splitting water takes a fixed energy, the
enthalpy $\Delta H$; at high temperature a larger part of it, $T\Delta S$, can be
supplied as heat and a smaller part, $\Delta G$, must be electricity. If the
steam is raised by heat from a neighbouring process, an SOEC needs about a
quarter less electricity per kilogram of hydrogen than a low-temperature
electrolyser. The electrode reactions are also fast when hot, so no platinum
or iridium is needed. The same stack can co-electrolyse steam and CO₂ to
syngas, or run in reverse as a fuel cell.

| | alkaline | PEM | solid oxide |
|---|---|---|---|
| electrolyte | KOH solution | polymer membrane | ceramic (zirconia) |
| ion through it | OH⁻ | H⁺ | O²⁻ |
| temperature | 60 – 90 °C | 50 – 80 °C | 700 – 850 °C |
| catalysts | nickel | platinum, iridium | nickel, ceramic oxides |
| electricity per kg H₂ | about 50 – 55 kWh | about 50 – 55 kWh | about 40 kWh, steam supplied |
| following a varying load | moderate | fast | slow; kept hot on standby |
| maturity | large plants | large plants | first plants of a few MW |

*Typical values from the literature, rounded; they vary with the stack and the
operating point.*

**The main challenges.**

- **Degradation.** Ceramic and metal layers age faster at 800 °C than in a
  low-temperature cell: nickel particles coarsen and move in the fuel
  electrode, the air electrode can delaminate at high current, and chromium
  from the steel interconnects poisons it. Stack lifetime is the main open
  question.
- **Heating and cycling.** A ceramic stack is heated slowly to avoid cracking
  and sealing failures, so it follows wind and solar poorly and is kept hot
  between uses. For the same reason the temperature may change only by a few
  kelvin along a cell in operation.
- **Steam and heat.** The plant needs a steam supply and heat exchangers; the
  efficiency gain is real only where heat is available.
- **Gas composition along the cell.** With too much steam and too little
  hydrogen the nickel oxidises (hence the 10 % hydrogen in the feed below);
  where the steam runs short towards the outlet, the local voltage loss rises
  and the electrode ages faster.

The cell below draws the whole cell; the rest of the notebook is about its top
layer, the hydrogen (fuel) channel, and the cell directly under it.

In [ ]:
pb.draw_soec(); plt.show()                                   # the whole cell: channel, electrodes, electrolyte, air side

**What you should see.** Steam goes down into the fuel electrode and hydrogen
comes back up into the channel, one for one; the oxide ions go down through
the electrolyte and leave as oxygen in the air channel; the power supply
drives the electrons into the fuel electrode.

### The hydrogen channel

A solid oxide cell with 100 × 100 mm of electrode runs as an electrolyser. A
mixture of 90 % steam and 10 % hydrogen enters its fuel channel at 1 m/s and
at the inlet temperature $T_{in}$. The cell is held at a voltage $V$. Along the
channel the steam is split and hydrogen takes its place, a current flows
through the cell at every point, and that current releases or absorbs heat.

**The question.** How do the steam fraction, the current density and the
temperature settle along the cell at a given cell voltage and inlet
temperature - and which cell voltage makes an hour of operation cheapest
without the temperature changing by more than 10 K along the cell?

## 2 · The data

| | |
|---|---|
| cell | 100 mm long, 100 mm wide: 100 cm² of electrode |
| fuel channel | Ex_09.2's channel: 1 mm high, gas at 1 m/s, 90 % steam at the inlet, 1 atm |
| operating range | cell voltage 1.15 to 1.35 V; inlet temperature 750 to 850 °C |
| ohmic resistance | 0.15 Ω cm² at 800 °C, activation energy 80 kJ/mol — **estimates** |
| electrodes | exchange current density 0.50 A/cm²; limiting current density 4.0 A/cm² at the inlet steam fraction — **estimates** |
| gas diffusivity | 8.0e-04 m²/s, as in Ex_09.2 — **an estimate** |
| heat | heat capacity flow of the gases 4.0 W/(m K) per m of cell width; in-plane conduction of the cell 0.025 W/K — **estimates** |
| prices | hydrogen 5 €/kg, electricity 0.05 €/kWh — **chosen for the exercise** |
| ageing | stack 2000 €/kW; end of life when the resistance has doubled; an Arrhenius law in temperature — **estimates** |
| thermoneutral voltage | $V_{tn}$ = 1.29 V |

The thermodynamics is standard. Every number marked as an estimate is of the
right order and was not measured on any cell; every result below depends on
them, and a result that depends on them must say so.

## 3 · The physics, and why it is multiphysics

Three fields along the channel, with $x$ the position over the channel length
(0 at the inlet, 1 at the outlet): the steam fraction $y$, the temperature $T$
(cell and gas at one value), and the local current density $i$ [A/cm²].

**The charge transfer** decides the current. At every point the cell voltage
is the sum of the Nernst potential and three losses (L10.2's *Polarisation
Curve*):

$$V = E(y, T) + \frac{RT}{F}\,\mathrm{asinh}\frac{i}{2 i_0} + i\,\mathrm{ASR}(T) - \frac{RT}{2F}\ln\Big(1 - \frac{i}{i_L(y)}\Big)$$

the Nernst potential $E$, the activation loss of the electrodes, the ohmic
loss through a resistance that falls as the cell gets hotter, and the
gas-transport loss, which grows without bound as the current approaches the
limiting current density $i_L$, the most steam the porous electrode can take
in (proportional to $y$). The cell voltage $V$ is fixed; **the current is
whatever satisfies this equation** - there is no derivative in it.

**The steam balance** is the convection–diffusion equation of L9.2 and
Ex_09.2 with a sink, the steam the current uses (Faraday's law). In scaled
units,

$$y_x = \frac{1}{\mathrm{Pe}}\,y_{xx} - K_I\, i, \qquad \mathrm{Pe} = \frac{uL}{D} = 125$$

**The energy balance** carries the heat along with the gases, conducts it
along the cell, and has a source: a current $i$ at voltage $V$ delivers
$iV$ of electrical power per unit area, of which $iV_{tn}$ goes into
splitting the steam, so the heat released is $i\,(V - V_{tn})$ - positive
above the thermoneutral voltage, negative below it. With
$\theta = (T - T_{in})/50\ \mathrm{K}$,

$$\theta_x = \frac{1}{\mathrm{Pe}_T}\,\theta_{xx} + S_Q\, i\,(V - V_{tn}), \qquad \mathrm{Pe}_T = \frac{\Gamma L}{k t} = 16$$

with $y = 0.90$ and $\theta = 0$ at the inlet and zero slopes at the outlet.

**Why it is multiphysics.** The cell below draws the three laws and what each
passes to the others. Four difficulties follow, and every method in this
notebook must handle them:

1. **Two-way coupling.** The current depends on the steam fraction and the
   temperature, and changes both. No field can be solved first and handed on.
2. **An algebraic equation inside a differential system.** The current has no
   equation of its own along $x$; at every point it is the root of the charge
   transfer, a nonlinear equation in $i$.
3. **Scales that differ.** The steam is carried 125 times faster than it
   diffuses, the heat 16 times faster than it is conducted; a temperature
   change of 50 K changes the ohmic resistance by a factor of 1.5 and the
   current by tens of per cent.
4. **Feedback through the temperature.** Above $V_{tn}$ the current heats the
   cell, the resistance falls and the current rises further; below $V_{tn}$
   the cell cools and the current falls. The temperature limit of section 6
   exists because of this.

In [ ]:
pb.describe_problem()
pb.draw_cell(); plt.show()                                   # the fuel channel and the cell under it
pb.draw_coupling(); plt.show()                               # the three laws and what each passes to the others

**What you should see.** The data of section 2 with the numbers they imply:
a Nernst potential of 0.85 V at the inlet at 800 °C, Pe = 125 and
Pe$_T$ = 16, and a steam supply that would take 197 A to use completely. The
coupling diagram: the charge transfer in the middle, taking the steam
fraction and the temperature, giving the current to both balances.

---

## 4 · The reference, and coupled finite volumes

**The reference.** There is no exact solution, so the methods are scored
against a **converged numerical solution**: SciPy's boundary-value solver, a
collocation method that refines its own mesh until the residual is below a
tolerance, with the charge transfer solved for the current at every point
(`pb.current`, bisection between zero and the limiting current).
`pb.reference(V, T_in)` returns a function of $x$ that gives $(y, T, i)$. The
cell below solves it at two tolerances, then draws the three fields at three
cell voltages at an inlet of 800 °C.

In [ ]:
T_IN = 1073.15                                                # K, 800 degC: the inlet temperature of sections 4 and 6
xs = np.linspace(0, 1, 401)                                   # the points everything is compared on
mm = xs * pb.L_CHANNEL * 1e3                                  # the same points in mm
t0 = time.perf_counter(); ref = pb.reference(1.29, T_IN); t_ref = time.perf_counter() - t0
y_r, T_r, i_r = ref(xs)
y_l, T_l, _ = pb.reference(1.29, T_IN, tol=1e-6)(xs)          # a looser tolerance, to see that the reference has converged
print(f"  tolerance 1e-06 against 1e-09: {np.abs(y_l - y_r).max():.1e} apart in steam fraction, "
      f"{np.abs(T_l - T_r).max():.1e} K in temperature   (one solve: {t_ref:.2f} s)")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for V, col in ((1.15, "tab:blue"), (1.29, "tab:green"), (1.35, "tab:red")):
    y, T, i = pb.reference(V, T_IN)(xs)
    axes[0].plot(mm, y, color=col, lw=2, label=f"{V:.2f} V")
    axes[1].plot(mm, T - 273.15, color=col, lw=2)
    axes[2].plot(mm, i, color=col, lw=2)
    c = pb.cost_per_hour(xs, i, T, V)
    print(f"  {V:.2f} V: steam fraction {y[-1]:.2f} at the outlet, temperature {T.min() - 273.15:.2f} to {T.max() - 273.15:.2f} degC, "
          f"current density {i[0]:.2f} -> {i[-1]:.2f} A/cm2, cell current {c['current']:.2f} A")
axes[0].set_ylabel("steam fraction  y  [-]"); axes[0].legend(frameon=False)
axes[1].set_ylabel("temperature  T  [°C]"); axes[2].set_ylabel("current density  i  [A/cm²]")
for a in axes:
    a.set_xlabel("x along the channel  [mm]")
plt.suptitle(f"the reference at an inlet of {T_IN - 273.15:.0f} °C", fontsize=10); plt.tight_layout(); plt.show()

**What you should see.** The two tolerances agree far better than the
accuracy agreed below, so this is the reference. The three voltages tell
three stories. At **1.15 V**, below the thermoneutral voltage, the cell
absorbs heat and cools by 27 K along the channel; its resistance
rises and the current density falls from 1.23 to 0.64 A/cm². At **1.29 V** the
heat almost balances and the temperature stays within 2 K of the
inlet. At **1.35 V** the cell heats by 23 K and uses more steam: the
current density falls along the channel because the steam runs short.

**Coupled finite volumes.** The channel is cut into $n$ cells. Each keeps the
balance of its steam and of its heat: what the flow brings in from upstream
(the upstream value, **upwind**), what diffusion and conduction exchange with
both neighbours, and what the current uses or releases. The current in each
cell is the root of the charge transfer at that cell's steam fraction and
temperature.

**Newton–Raphson.** The $2n$ balances, collected as $F(u) = 0$ with $u$ the
$2n$ unknowns (the steam fraction and the temperature in every cell), are
nonlinear: the current depends on them through the Nernst logarithm, the
activation loss and the resistance. They are solved together by the
Newton–Raphson method. Starting from the inlet values everywhere, each step
linearises the balances about the current guess and solves one sparse linear
system,

$$J(u)\,\Delta u = -F(u), \qquad u \leftarrow u + \Delta u,$$

until the update $\Delta u$ is below $10^{-10}$. $J$ is the Jacobian, the
derivative of every balance with respect to every unknown. It has four
blocks: steam–steam and heat–heat, tridiagonal (convection and diffusion
between neighbours), and two diagonal **coupling blocks**, the steam balance
of a cell against its temperature and its heat balance against its steam
fraction. Both come from the derivatives of the current, obtained by
differentiating the charge transfer (the implicit-function rule):
$\partial i/\partial y = -(\partial V/\partial y)/(\partial V/\partial i)$,
and the same for $T$. Near the solution Newton–Raphson converges
quadratically, each step roughly doubling the number of correct digits, so a
handful of steps suffices. Solved one field at a time (steam with the
temperature frozen, then the temperature with the current frozen) the
coupling would converge only linearly and take many more passes. Inside each
step the current in every cell is found by bisection between zero and the
limiting current, which cannot leave the range where the gas-transport
logarithm is defined.

**The agreed accuracy is 0.01 in steam fraction and 0.5 K in temperature**,
anywhere along the channel. The cell below finds the coarsest grid that meets
both at 1.35 V and 800 °C, then shows what the coupling is worth: the same
cell solved with the heat left out, the temperature held at the inlet value.

In [ ]:
def fvm(V, T_in, n=50, heat=True, tol=1e-10):
    # coupled finite volumes: steam fraction y and scaled temperature theta in n cells, Newton-Raphson on all 2n unknowns
    h = 1.0 / n; xc = (np.arange(n) + 0.5) * h                # cell width and cell centres
    dy, dt = 1 / (pb.PE * h * h), 1 / (pb.PE_T * h * h)       # diffusion and conduction between neighbouring cells
    a_in = np.ones(n); a_in[0] = 2.0                          # the inlet face is half a cell away
    a_out = np.ones(n); a_out[-1] = 0.0                       # zero slope at the outlet: nothing leaves by diffusion
    q = pb.S_Q * (pb.V_TN - V) * heat                         # heat absorbed per unit current, scaled (zero: heat left out)
    y, th, e = np.full(n, pb.Y_IN), np.zeros(n), 1e-7
    for k in range(30):                                       # Newton-Raphson steps: J du = -F, u <- u + du
        T = T_in + pb.DT_SCALE * th
        i = pb.current(V, y, T)                               # the charge transfer, solved in every cell
        dV_di = (pb.cell_voltage(i + e, y, T) - pb.cell_voltage(i - e, y, T)) / (2 * e)
        i_y = -(pb.cell_voltage(i, y + e, T) - pb.cell_voltage(i, y - e, T)) / (2 * e) / dV_di      # di/dy
        i_th = -(pb.cell_voltage(i, y, T + 1e-4) - pb.cell_voltage(i, y, T - 1e-4)) / 2e-4 / dV_di * pb.DT_SCALE * heat  # di/dtheta
        yu, yd = np.r_[pb.Y_IN, y[:-1]], np.r_[y[1:], y[-1]]  # upstream and downstream neighbours
        tu, td = np.r_[0.0, th[:-1]], np.r_[th[1:], th[-1]]
        r_y = (y - yu) / h - dy * (a_out * (yd - y) - a_in * (y - yu)) + pb.K_I * i      # steam balance of each cell
        r_t = (th - tu) / h - dt * (a_out * (td - th) - a_in * (th - tu)) + q * i        # heat balance of each cell
        J_yy = sp.diags([-(1 / h + dy * a_in[1:]), 1 / h + dy * (a_out + a_in) + pb.K_I * i_y, -dy * a_out[:-1]], [-1, 0, 1])
        J_tt = sp.diags([-(1 / h + dt * a_in[1:]), 1 / h + dt * (a_out + a_in) + q * i_th, -dt * a_out[:-1]], [-1, 0, 1])
        J = sp.bmat([[J_yy, sp.diags(pb.K_I * i_th)],          # the coupling blocks: the steam through the temperature,
                     [sp.diags(q * i_y), J_tt]], format="csc")   # the heat through the steam
        step = spla.spsolve(J, -np.r_[r_y, r_t])              # the Newton-Raphson update du
        y, th = y + step[:n], th + step[n:]
        if np.abs(step).max() < tol:
            break
    T = T_in + pb.DT_SCALE * th
    return xc, y, T, pb.current(V, y, T), k + 1

TOL_Y, TOL_T = 0.01, 0.5                                      # the agreed accuracy: steam fraction, kelvin
ref_hard = pb.reference(1.35, T_IN)
for n in (25, 50, 100, 200):
    t0 = time.perf_counter(); xc, y_f, T_f, _, its = fvm(1.35, T_IN, n); sec = time.perf_counter() - t0
    y_e, T_e, _ = ref_hard(xc)
    ey, eT = np.abs(y_f - y_e).max(), np.abs(T_f - T_e).max()
    print(f"  {n:3d} cells: worst error {ey:.4f} in steam fraction, {eT:.2f} K; {its} Newton-Raphson steps, {1000 * sec:.1f} ms"
          + ("   <- meets both" if ey <= TOL_Y and eT <= TOL_T else ""))
N_FVM = 50                                                    # the coarsest grid that meets both

print("what the heat is worth, 50 cells, inlet at 800 degC:")
for V in (1.15, 1.29, 1.35):
    xc, y_c, T_c, i_c, _ = fvm(V, T_IN, N_FVM)
    _, y_0, _, i_0, _ = fvm(V, T_IN, N_FVM, heat=False)       # the temperature held at the inlet value
    I_c, I_0 = np.mean(i_c) * pb.A_CELL, np.mean(i_0) * pb.A_CELL
    print(f"  {V:.2f} V: cell current {I_c:.2f} A coupled, {I_0:.2f} A with the heat left out "
          f"({100 * (I_0 - I_c) / I_c:+.1f} %); steam fraction at the outlet {y_c[-1]:.2f} against {y_0[-1]:.2f}")

**What you should see.** The error falls with every doubling of the cells,
at first order (the price of the upwind value), and **50 cells are the first
to meet both 0.01 and 0.5 K**, in five Newton–Raphson steps and some 25
milliseconds; it meets them at every operating point of the range (section 5
checks twelve). Leaving the heat out changes the answer most where the cell is
furthest from thermoneutral: at 1.15 V the uncoupled model overstates the
cell current by 8 %, because it does not see the cell cool and its
resistance rise; at 1.29 V the two agree. **The coupling cannot be dropped**,
and a method for this cell must solve all three laws together.

---

## 5 · Part A: one network for every operating point

**What the network is for.** Finite volumes answer one operating point at a
time, in milliseconds. A design study, a controller or an optimiser asks for
thousands: every cell voltage, every inlet temperature. The network here is
trained **once** over the whole operating range and then answers any
operating point by one evaluation.

**The network** takes three inputs, each scaled to [0, 1]: the position $x$,
the cell voltage $v$ (1.15 to 1.35 V) and the inlet temperature $\tau$ (750
to 850 °C). It returns the three fields. Four layers of 64 tanh neurons, three
outputs $\mathcal{N}_1, \mathcal{N}_2, \mathcal{N}_3$, and a trial function
that builds in what is known exactly:

$$y = 0.90 + x\,\mathcal{N}_1, \qquad \theta = x\,\mathcal{N}_2, \qquad i = i_L(y)\,\sigma(\mathcal{N}_3)$$

The inlet values hold at every operating point (hard enforcement), and the
sigmoid $\sigma$ keeps the current density between zero and the limiting
current density, the only range where the charge transfer has a root. The
zero slopes at the outlet are a loss term (soft enforcement).

**The loss** is the mean square of three residuals, one per law, at points
spread over the channel and the operating range, plus the outlet slopes:

* the steam balance, $r_y = y_x - y_{xx}/\mathrm{Pe} + K_I\,i$
* the energy balance, $r_\theta = \theta_x - \theta_{xx}/\mathrm{Pe}_T - S_Q\,i\,(V - V_{tn})$
* the charge transfer, $r_c = \big(V_{\mathrm{cell}}(i, y, T) - V\big) / 0.05\ \mathrm{V}$

The charge transfer is where the coupling enters the loss: the network's
current density must make the cell's own voltage equal the applied voltage at
every point. That residual is in volts and is divided by **50 mV**, so that an
error of 50 mV weighs as much as an error of one in a scaled balance; without
the scaling, millivolt errors would be invisible beside the balances. A fifth
of the points lie on the edges of the operating range, where the cell runs
shortest of steam or cools most.

**TODO 1** is the two balances; **TODO 2** the charge transfer.

In [ ]:
class Cell(torch.nn.Module):                                  # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_out=3, n_hidden=64, n_layers=4)   # x, v, tau in; three outputs
    def forward(self, p):
        o = self.net(2 * p - 1)                               # inputs centred on [-1, 1] for tanh
        x = p[:, 0:1]
        y = pb.Y_IN + x * o[:, 0:1]                           # steam fraction: the inlet value, exactly
        th = x * o[:, 1:2]                                    # scaled temperature: zero at the inlet, exactly
        i = pb.i_lim(y) * torch.sigmoid(o[:, 2:3])            # current density between zero and its limit, A/cm2
        return y, th, i

def operating_point(p):
    # the cell voltage [V] and the inlet temperature [K] at each point
    return pb.voltage(p[:, 1:2]), pb.inlet_temperature(p[:, 2:3])

In [ ]:
def balances(model, p):
    y, th, i = model(p)                                       # the three fields at the points
    V, T_in = operating_point(p)
    # TODO 1: the residuals of the steam balance and of the energy balance.
    # grad(u, p)[:, 0:1] is the slope along the channel; d2(u, p, 0) the second derivative.
    #
    #   y_x, th_x = grad(y, p)[:, 0:1], grad(th, p)[:, 0:1]
    #   r_y = y_x - d2(y, p, 0) / pb.PE + pb.K_I * i
    #   r_t = th_x - d2(th, p, 0) / pb.PE_T - pb.S_Q * i * (V - pb.V_TN)
    #   return r_y, r_t
    raise NotImplementedError("Write the two balances")

In [ ]:
def charge_transfer(model, p):
    y, th, i = model(p)                                       # the three fields at the points
    V, T_in = operating_point(p)
    # TODO 2: the residual of the charge transfer, scaled by 50 mV.
    # pb.cell_voltage(i, y, T) takes the temperature in kelvin.
    #
    #   T = T_in + pb.DT_SCALE * th
    #   return (pb.cell_voltage(i, y, T) - V) / 0.05
    raise NotImplementedError("Write the charge transfer")

In [ ]:
def loss_fn(model, p, q):
    # the composite loss at collocation points p and outlet points q
    r_y, r_t = balances(model, p)
    r_c = charge_transfer(model, p)
    y, th, _ = model(q)
    slopes = grad(y, q)[:, 0:1].pow(2).mean() + grad(th, q)[:, 0:1].pow(2).mean()   # zero slopes at the outlet
    parts = dict(steam=r_y.pow(2).mean(), heat=r_t.pow(2).mean(), charge=r_c.pow(2).mean(), outlet=slopes)
    return sum(parts.values()), parts

def points(n, seed, outlet=False):
    # n points (x, v, tau), differentiable; a fifth on the edges of the operating range
    p = pb.sample_points(n, seed=seed)
    if outlet:
        p[:, 0] = 1.0                                         # the outlet, at the same spread of operating points
    return to_tensor(p, requires_grad=True)                   # differentiable: the residuals take slopes in x

def train(model, adam_steps, lbfgs_iter=0, report=500):
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)      # Adam first: robust far from a solution
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, adam_steps, 1e-5)   # the step shrinks to 1e-5
    for k in range(adam_steps):
        p, q = points(3000, seed=k), points(256, seed=10**6 + k, outlet=True)   # new points every step
        loss, parts = loss_fn(model, p, q)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()   # gradients by autograd, one step, a smaller step size
        if report and k % report == 0:
            print(f"    adam {k:5d}  loss {loss.item():.2e}  " + "  ".join(f"{a} {b.item():.1e}" for a, b in parts.items()))
    if lbfgs_iter:
        p, q = points(4000, seed=123456), points(256, seed=2 * 10**6, outlet=True)   # fixed points: L-BFGS needs them
        opt = torch.optim.LBFGS(model.parameters(), lr=1, max_iter=lbfgs_iter, history_size=50, line_search_fn="strong_wolfe")   # then L-BFGS: curvature to polish
        def closure():
            opt.zero_grad(); loss, _ = loss_fn(model, p, q); loss.backward(); return loss   # L-BFGS re-evaluates the loss in its line search
        opt.step(closure)                                     # up to lbfgs_iter iterations on the fixed points
        print(f"    l-bfgs  loss {loss_fn(model, p, q)[0].item():.2e}")

FULL_TRAINING = False                                         # True: train the full 9000 + L-BFGS steps and save the weights
model = Cell()
describe(model.net, n_collocation=3000)                       # the parameter count and the collocation-to-parameter ratio
t0 = time.perf_counter()
train(model, 9000 if FULL_TRAINING else 1000, lbfgs_iter=3000 if FULL_TRAINING else 0)   # 1000 steps here, the saved weights 9000 and L-BFGS
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")
if FULL_TRAINING:
    torch.save(model.state_dict(), "Ex10.2_weights.pt")              # the weights the notebook loads otherwise

In [ ]:
def pinn_fields(V, T_in, x=xs):
    # the network's steam fraction, temperature [K] and current density [A/cm2] along the channel at one operating point
    v, tau = pb.scaled(V, T_in)
    with torch.no_grad():  # no gradient tracking: evaluation only
        y, th, i = model(to_tensor(np.c_[x, np.full(len(x), v), np.full(len(x), tau)]))   # one operating point at every x
    return to_numpy(y).ravel(), T_in + pb.DT_SCALE * to_numpy(th).ravel(), to_numpy(i).ravel()

GRID = [(V, T) for V in (1.15, 1.22, 1.29, 1.35) for T in (1023.15, 1073.15, 1123.15)]   # 12 operating points
REFS = {op: pb.reference(*op) for op in GRID}                # the reference at each, a function of x

def score(fields):
    # the worst errors over the 12 operating points, each method at its own points: steam fraction, kelvin, A/cm2
    e = []
    for op in GRID:
        x, *got = fields(*op)
        e.append([np.abs(a - b).max() for a, b in zip(got, REFS[op](x))])
    return np.max(e, axis=0)

pinn_at = lambda V, T_in: (xs, *pinn_fields(V, T_in))           # the network at the 401 comparison points
e_short = score(pinn_at)
print(f"trained in this session:  worst error {e_short[0]:.4f} in steam fraction, {e_short[1]:.2f} K, {e_short[2]:.3f} A/cm2")
if not FULL_TRAINING:
    model.load_state_dict(torch.load("Ex10.2_weights.pt", map_location=DEVICE))   # the same network, trained 9000 Adam steps and L-BFGS
e_pin = score(pinn_at)
print(f"the trained network:      worst error {e_pin[0]:.4f} in steam fraction, {e_pin[1]:.2f} K, {e_pin[2]:.3f} A/cm2")
e_fvm = score(lambda V, T_in: fvm(V, T_in, N_FVM)[:4])          # finite volumes at their own cell centres
print(f"finite volumes, 50 cells: worst error {e_fvm[0]:.4f} in steam fraction, {e_fvm[1]:.2f} K, {e_fvm[2]:.3f} A/cm2")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for (V, T_in), col in (((1.15, 1023.15), "tab:blue"), ((1.29, 1073.15), "tab:green"), ((1.35, 1123.15), "tab:red")):
    for a, r, n in zip(axes, REFS[(V, T_in)](xs), pinn_fields(V, T_in)):
        k = 273.15 if a is axes[1] else 0                     # degC on the temperature panel
        a.plot(mm, r - k, color=col, lw=2.5, label=f"reference, {V:.2f} V, {T_in - 273.15:.0f} °C")
        a.plot(mm, n - k, "k:", lw=1.8)
axes[0].plot([], [], "k:", label="network"); axes[0].legend(frameon=False, fontsize=8)
axes[0].set_ylabel("steam fraction  y  [-]"); axes[1].set_ylabel("temperature  T  [°C]"); axes[2].set_ylabel("current density  i  [A/cm²]")
for a in axes:
    a.set_xlabel("x along the channel  [mm]")
plt.tight_layout(); plt.savefig(cc.output_path("Ex10.2_report_1.png"), dpi=150); plt.show()

**What you should see.** The network trained in this session has learnt the
shape but is not yet at the agreed accuracy. The trained network - the same
code, 9000 Adam steps and L-BFGS, saved in `Ex10.2_weights.pt` - is within 0.0022 of the
steam fraction, 0.19 K and 0.016 A/cm² **at all twelve operating
points**, corners included, where finite volumes on 50 cells are within
0.0068, 0.40 K and 0.023 A/cm²: the network is about three times more
accurate than the grid the agreed accuracy asks for. The dotted lines lie on the references
at the coolest, the middle and the hottest corner.

**The comparison that matters: many operating points.** The cell below asks
both methods for the three fields at 1000 random operating points. Finite
volumes solve them one after another; the network evaluates them in one
batch. Both run on this runtime; on a GPU it is synchronised before the clock
is read. Only the online cost is timed; the training is the offline cost,
reported apart.

In [ ]:
rng = np.random.default_rng(1)
OPS = np.c_[rng.uniform(pb.V_LOW, pb.V_HIGH, 1000), rng.uniform(pb.T_LOW, pb.T_HIGH, 1000)]   # 1000 operating points
x51 = np.linspace(0, 1, 51)
def all_fvm():
    for V, T_in in OPS:
        fvm(V, T_in, N_FVM)
v_all, tau_all = pb.scaled(OPS[:, 0], OPS[:, 1])
P = to_tensor(np.c_[np.tile(x51, 1000), np.repeat(v_all, 51), np.repeat(tau_all, 51)])   # 51 000 points, one batch
def all_pinn():
    with torch.no_grad():                                     # no gradient tracking: evaluation only
        model(P)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()                              # wait for the GPU before the clock is read
t0 = time.perf_counter(); all_fvm(); t_fvm = time.perf_counter() - t0
t_nn = middle(all_pinn, reps=5)
print(f"1000 operating points on {DEVICE.type}: finite volumes {t_fvm:.2f} s ({t_fvm:.1f} ms each), "
      f"network {1000 * t_nn:.1f} ms in one batch: {t_fvm / t_nn:.0f} times faster")
TRAIN_FULL_S = 2005                                     # s: the full training of the saved weights, measured on a 2-core CPU
print(f"offline: the full training took {TRAIN_FULL_S / 60:.0f} min on a 2-core CPU; "
      f"repaid after about {TRAIN_FULL_S / (t_fvm / 1000 - t_nn / 1000):.0f} operating points")

**What you should see.** Finite volumes take 25 ms per operating point,
because every Newton–Raphson step solves the charge transfer in every cell; the
network answers all 1000 in 50 ms - **about 500 times faster**
on a CPU. The training is repaid after some 80 000 operating points. For
one operating point, finite volumes are the tool; for many, the network.

---

## 6 · Part B: the optimal cell voltage

**The cost of an hour.** For one cell, at cell voltage $V$ and current
$I = \int i\,dA$ [A],

$$C(V) = p_{el}\,V I \;-\; p_{H_2}\,\frac{I\,M_{H_2}}{2F} \;+\; \frac{c_{stack}\,P_{rated}}{\Delta \mathrm{ASR}_{EOL}}\int k_0\,e^{-E_d/RT}\,i^{1.5}\,dA$$

the electricity bought, less the hydrogen sold, plus the share of the stack
that ages in the hour: the ageing law (an empirical Arrhenius law, as in
L10.2) gives the growth of the resistance per hour, and the stack is
replaced when its resistance has doubled. A negative cost is a profit. A
higher voltage makes more hydrogen but costs more electricity per kilogram
and ages the cell faster.

**The constraint.** The temperature may change by at most 10 K along the cell,
$\max_x |T - T_{in}| \le 10\ \mathrm{K}$, to protect the ceramic layers.

**The optimisation.** The network is a formula for $(y, T, i)$ in terms of
$V$, so $C(V)$ and the largest temperature change are formulas in $V$, and
**autograd gives their derivatives**. The cell below minimises
$C(V) + \mu\,\max(0, \Delta T - 10\ \mathrm{K})^2$, the constraint as a
penalty, by gradient descent on $V$ at an inlet of 800 °C - once without the
constraint ($\mu = 0$) and once with it - and checks both against a sweep of
81 voltages solved by finite volumes on 200 cells.

In [ ]:
for p_ in model.parameters():
    p_.requires_grad_(False)                                  # the network is finished: only the voltage moves now
xq = to_tensor(np.linspace(0, 1, 101).reshape(-1, 1))         # 101 points along the channel, for the integrals
tau = float(pb.scaled(0.0, T_IN)[1])                          # the scaled inlet temperature, 800 degC

def cost_nn(V):
    # the cost of an hour [cent/h] and the largest temperature change [K] at cell voltage V, through the network
    v = (V - pb.V_LOW) / (pb.V_HIGH - pb.V_LOW)
    y, th, i = model(torch.cat([xq, v.expand(len(xq), 1), torch.full_like(xq, tau)], 1))
    T = T_IN + pb.DT_SCALE * th
    return pb.cost_per_hour(xq, i, T, V)["total"].squeeze(), (pb.DT_SCALE * th).abs().max()

OPT = {}
for mu, label in ((0.0, "free"), (5.0, "dT <= 10 K")):
    V = torch.tensor(1.25, device=DEVICE, requires_grad=True) # start at 1.25 V
    opt = torch.optim.Adam([V], lr=2e-3)                       # an optimiser for one number
    t0 = time.perf_counter()
    for step in range(400):
        c, dT = cost_nn(V)
        J = c + mu * torch.relu(dT - pb.DT_MAX) ** 2           # the cost, with the penalty where the limit is passed
        opt.zero_grad(); J.backward(); opt.step()              # dJ/dV by autograd through the network, then one step down
        with torch.no_grad():                                 # change the voltage itself, not through the graph
            V.clamp_(pb.V_LOW, pb.V_HIGH)                     # stay inside the range the network was trained on
    c, dT = cost_nn(V)
    OPT[label] = (V.item(), c.item(), dT.item(), time.perf_counter() - t0)
    print(f"network, {label:10s}: V* = {V.item():.3f} V, cost {c.item():.2f} cent/h, largest change {dT.item():.2f} K   "
          f"({OPT[label][3]:.1f} s, 400 steps)")

t0 = time.perf_counter()
Vs = np.linspace(pb.V_LOW, pb.V_HIGH, 81); SW = []              # the check: finite volumes at 81 voltages
for V in Vs:
    xc, y, T, i, _ = fvm(V, T_IN, 200)
    c = pb.cost_per_hour(xc, i, T, V); c["dT"] = np.abs(T - T_IN).max(); SW.append(c)
t_sweep = time.perf_counter() - t0
cost = np.array([c["total"] for c in SW]); dTs = np.array([c["dT"] for c in SW])
k_free = int(np.argmin(cost)); ok = np.where(dTs <= pb.DT_MAX)[0]; k_con = int(ok[np.argmin(cost[ok])])
print(f"81 voltages by finite volumes ({t_sweep:.1f} s): free V* = {Vs[k_free]:.3f} V ({dTs[k_free]:.2f} K), "
      f"with dT <= 10 K V* = {Vs[k_con]:.3f} V, cost {cost[k_con]:.2f} cent/h ({dTs[k_con]:.2f} K)")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for key, col, lab in (("total", "k", "total"), ("electricity", "tab:orange", "electricity"),
                      ("ageing", "tab:red", "ageing of the stack")):
    axes[0].plot(Vs, [c[key] for c in SW], color=col, lw=2.5 if key == "total" else 1.5, label=lab)
axes[0].plot(Vs, [-c["hydrogen"] for c in SW], color="tab:green", lw=1.5, label="hydrogen sold (negative)")
for label, m in (("free", "o"), ("dT <= 10 K", "s")):
    axes[0].plot(OPT[label][0], OPT[label][1], m, color="tab:blue", ms=9, label=f"network optimum, {label}")
axes[0].set_xlabel("cell voltage  V  [V]"); axes[0].set_ylabel("cost of an hour  [cent/h]"); axes[0].legend(frameon=False, fontsize=8)
axes[1].plot(Vs, dTs, "k", lw=2); axes[1].axhline(pb.DT_MAX, color="tab:red", ls="--", label="limit, 10 K")
axes[1].axvline(pb.V_TN, color="0.5", ls=":", label="thermoneutral voltage")
axes[1].set_xlabel("cell voltage  V  [V]"); axes[1].set_ylabel("largest temperature change  [K]"); axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(cc.output_path("Ex10.2_report_2.png"), dpi=150); plt.show()

**What you should see.** Without the constraint the cost is lowest at about
1.167 V, below the thermoneutral voltage: little electricity per kilogram
of hydrogen and a cool, slowly ageing cell - but the cell cools by
25 K along its length, more than the ceramic layers allow. With the
constraint the optimum moves up to about 1.250 V, where the cooling is just
10 K; the hour then earns 0.46 cent against 0.54 at the free optimum
(the cost is negative: a profit). The network's gradient and the sweep of
finite volumes agree on both. The left panel shows why the free optimum is so
low: with these estimates the ageing of the stack costs more than the
electricity, and it grows with the current. Near the constrained optimum the temperature
change depends on the voltage much more strongly than the cost does.

---

## 7 · What the notebook says

* **The cell is multiphysics.** The current density is the root of the
  charge transfer at every point; it depends on the steam fraction and the
  temperature and changes both. Leaving the heat out overstated the cell
  current at 1.15 V by 8 %.
* **For one operating point, finite volumes are the tool.** 50 cells meet the
  agreed accuracy in a few milliseconds and need no training.
* **For many operating points, the network pays.** One network, trained once
  over the cell voltage and the inlet temperature, answered 1000 operating
  points about 500 times faster, as accurate as 50 cells everywhere in
  the range. Its training is repaid after some 80 000 answers.
* **The network gives a gradient.** The optimal voltage under a temperature
  limit came from gradient descent through it, in seconds. With one number to
  choose, a sweep of the solver does as well; with a voltage for every hour of
  a day, or for many cells, it does not (mini project B).
* **The temperature limit decides the voltage.** The cheapest operation is
  below thermoneutral, where the cell cools; the limit moves it towards
  thermoneutral.
* **The numbers are only as good as their estimates.** The kinetics, the heat
  transport, the ageing law and the prices were chosen, not measured. The
  method carries over to a real cell; these numbers do not.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex10.2_report.md` with the results of section 5 and 6 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L10.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The current density is not given but solved at every point. Which fields does it depend on, which does it change, and what did leaving the heat out change at 1.15 V? (-> L10.2 Q2)": """
""",
    "What is built into the network's form, what is held by the loss, why is the charge-transfer residual divided by 50 mV, and how close did the network come to the reference over the operating range? (-> L10.2 Q1)": """
""",
    "For 1000 operating points, how much faster was the network than finite volumes, on which runtime, and what does that comparison leave out? (-> L10.2 Q1)": """
""",
    "Which cell voltage minimised the cost of an hour with and without the 10 K limit, why do the two differ, and how were they checked? (-> L10.2 Q3)": """
""",
    "Which of the notebook's numbers would you refuse to promise for a real cell, and how does the ageing of the stack enter the choice of voltage? (-> L10.2 Q4)": """
""",
    "To conclude: for which task in this notebook would you use finite volumes, for which the network, and what changes when a voltage must be chosen for every hour of a day? (-> L10.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | worst error: steam fraction | temperature | current density | 1000 operating points | training |", "|---|---|---|---|---|---|",
            f"| finite volumes, 50 cells | {e_fvm[0]:.4f} | {e_fvm[1]:.2f} K | {e_fvm[2]:.3f} A/cm² | {t_fvm:.2f} s | - |",
            f"| PINN, 4 x 64, every operating point | {e_pin[0]:.4f} | {e_pin[1]:.2f} K | {e_pin[2]:.3f} A/cm² | {t_nn:.3f} s | {TRAIN_FULL_S / 60:.0f} min offline |",
            "", f"Optimal cell voltage at an inlet of 800 °C: {OPT['free'][0]:.3f} V free ({OPT['free'][2]:.2f} K change), "
                f"{OPT['dT <= 10 K'][0]:.3f} V with the 10 K limit ({OPT['dT <= 10 K'][1]:.2f} cent/h); "
                f"finite-volume sweep {Vs[k_free]:.3f} V and {Vs[k_con]:.3f} V"]
    out = ["# Ex_10.2 — Steam, Charge and Heat in a Solid Oxide Electrolysis Cell", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex10.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5 and 6, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex10.2_report.md into Ex10.2_report.pdf, with any figure
# saved as Ex10.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex10.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex10.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex10.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex10.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP10.2A · The channel across its height, and the electrode under it

**The problem.** The notebook's channel is one-dimensional: one steam fraction at each $x$,
and a gas-transport loss that rests on an estimated limiting current density,
4 A/cm² at the inlet. In the cell the steam must cross the 1 mm channel to
the electrode surface and then diffuse through the porous fuel electrode to
the reaction at the electrolyte. Here both are resolved: the steam fraction
across the channel, with Poiseuille flow (Ex_09.2), and through the
electrode, and the charge transfer is evaluated at the reaction site. The
limiting current is no longer an input; it follows from the electrode.

| | |
|---|---|
| cell | the notebook's cell at 800 °C, **isothermal** (the heat was the notebook's coupling) |
| channel | 1 mm high, Poiseuille flow, mean 1 m/s, 90 % steam at the inlet; axial diffusion dropped (Pe 125) |
| fuel electrode | 0.5 mm, effective diffusivity of steam 1.0e-05 m²/s, no flow; the reaction at the electrolyte — **estimates** |
| charge transfer | $V = E(y_r, T) + \frac{RT}{F}\mathrm{asinh}\frac{i}{2i_0} + i\,\mathrm{ASR}$, with $y_r$ the steam fraction at the reaction site; no gas-transport term |
| voltage | 1.20, 1.29 and 1.35 V |

The electrode's diffusivity is chosen so that its limiting current, 3.94 A/cm²
at the inlet, is close to the notebook's 4.00; the difference between the
two models is then the physics, not the number.

**Why a plain PINN is not enough.** Two subdomains with different diffusivities meet at the electrode surface,
where the flux must be continuous; the current density is set at the wall by
a nonlinear equation in the steam fraction at the reaction site, and that
steam fraction is the answer, not an input. The wall row of the channel has
no flow, which a marching scheme must treat with care (Ex_09.2).

**What you build.** A PINN for the steam fraction $y(x, Z)$ across the channel, with the current
density $i(x)$ as a second output, trained on the convection–diffusion
residual, the wall flux $\partial y/\partial Z \propto i$, the steam drop
across the electrode, and the charge transfer at the reaction site; the inlet
built in. One network over the three voltages, or over a range of them. Then
the comparison with the notebook's one-dimensional model at the same voltages.

**The ground truth you get.** `MP10.2A_truth.npz`, from `tools/miniprojects/ex102_truth.py A`: the march
along the channel by Crank–Nicolson after two fully implicit steps, Newton at
every step, on 400 × 41, 800 × 81 and 1600 × 161 nodes; the steam fraction
over $(x, Z)$, the current density and the mixing-cup mean at each voltage.

**Worked example.** 

| voltage | total current | mean steam at the outlet | at the reaction site | the notebook's 1-D model |
|---|---|---|---|---|
| 1.20 V | 100.28 A | 0.4424 | 0.2502 | 109.05 A (+8.7 %) |
| 1.29 V | 128.13 A | 0.3153 | 0.0884 | 136.59 A (+6.6 %) |
| 1.35 V | 143.71 A | 0.2442 | 0.0231 | 150.72 A (+4.9 %) |

Across the channel the steam fraction changes by only 0.002 (the electrode
surface against the mean), as in Ex_09.2; across the electrode it falls by
0.19 to 0.23. The one-dimensional model overstates the current by 5 to 9 %,
because its gas-transport term counts the steam drop but not the hydrogen
that gathers at the reaction site. The grids 800 × 81 and 1600 × 161 agree to
0.005 A.

**Requirements.**

* The total current within **0.5 %** and the mean steam fraction at the outlet within **0.005**, at all three voltages.
* The steam fraction at the reaction site within **0.01** along the channel.
* Say where the steam fraction changes most - along the channel, across it, or through the electrode - and what that means for the notebook's model.

**What you hand in.** The steam fraction over the channel and the electrode at 1.29 V, the current
density along the channel against the ground truth's at the three voltages,
and the comparison with the notebook's one-dimensional model.

### MP10.2B · A day of operation under an hourly price

**The problem.** The notebook found one optimal voltage for one price. A plant buys its
electricity at a price that changes every hour, and it sells its hydrogen
under a contract for the day. Here the cell voltage and the inlet temperature
are chosen for each of 24 hours, 48 numbers in all, to make the contract at
the least cost of electricity and ageing, with the temperature change along
the cell at most 10 K in every hour.

| | |
|---|---|
| cell | the notebook's cell, its coupled model and its cost of an hour without the hydrogen sold |
| contract | 109.10 g of hydrogen a day: what 1.25 V at 800 °C makes in 24 hours |
| price | hourly, 0.02 €/kWh at midday to 0.14 €/kWh at 18:00 - **invented for the exercise** |
| decisions | $V$ from 1.15 to 1.35 V and $T_{in}$ from 750 to 850 °C, each hour |
| case 1 | the temperature change along the cell at most 10 K |
| case 2 | no temperature limit |

**Why a plain PINN is not enough.** Forty-eight numbers, each hour coupled to the others through the contract.
A sweep of finite volumes over 48 dimensions is out of the question; the
notebook's network, with the inlet temperature as an input, gives the cost
and the hydrogen of every hour and their gradients with respect to all 48 at
once.

**What you build.** The notebook's network (or your own, trained over the voltage and the inlet
temperature) as the model of every hour; the day's cost, the contract as a
constraint (a penalty or a Lagrange multiplier), the temperature limit as a
penalty; gradient descent on the 48 numbers. Then the comparison with the
best operating point held all day.

**The ground truth you get.** `MP10.2B_truth.npz`, from `tools/miniprojects/ex102_truth.py B`: the
notebook's coupled finite volumes (100 cells) on 81 × 41 operating points
(2.5 mV × 2.5 K); with the hydrogen priced by a Lagrange multiplier the hours
are independent, and the multiplier that makes the contract is found by
bisection. The schedule, its cost and the best constant point, both cases.

**Worked example.** Held at 1.25 V and 800 °C all day, the contract costs 50.37 cent. The best
constant point that makes it, with the 10 K limit, is 1.3175 V at 757.5 °C:
43.91 cent. The hourly schedule costs 43.53 cent, **0.9 % less** than that
point: it runs at about 1.32 V nearly all day and follows the price mostly
through the inlet temperature, warmer at midday and down to 750 °C in the
evening peak, where the voltage also falls to 1.28 V. Without the limit
(case 2), the schedule saves 2.9 % against its own best constant point, and
makes 2.86 to 4.87 g of hydrogen an hour against 3.89 to 4.91 with it: the
temperature limit takes most of the flexibility. In both cases the cheapest
operation is cool and hard, because with these estimates the ageing of the
stack costs more than the electricity, and it falls with temperature.

**Requirements.**

* Case 1: the day's cost within **0.5 %** of the ground truth, the contract met within **0.5 %**, and every hour within the 10 K limit by the finite volumes, not only by the network.
* Case 2: the same, and the saving against the best constant point within **0.5 percentage points**.
* Say which estimate decides that the cell runs cool, and how the schedule changes if the ageing cost is halved.

**What you hand in.** The 24-hour schedule of voltage and inlet temperature with its cost against
the ground truth's, checked by finite volumes hour by hour, in both cases, and
the effect of halving the ageing cost.